In [1]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

from torch.utils.data import DataLoader, TensorDataset

In [2]:
import sys
import os
sys.path.append(os.path.abspath("../"))

from src.lyap import NN_LyapExp
from src.networks import NeuralODEClassifier, NeuralODE_Truncated, init_weights
from src.utils import lyapunov_autograd, train

ImportError: cannot import name 'lyapunov_autograd' from 'src.utils' (c:\Users\m.tonti\Documents\lyap_exp\src\utils.py)

In [ ]:
from matplotlib.colors import LinearSegmentedColormap

tf_playground = LinearSegmentedColormap.from_list(
    "tf_playground",
    [
        "#f4a261", # orange
        "#ffffff", # white
        "#2a9df4"  # blue
    ]
)

In [ ]:
data = np.load("circle_dataset_eps0p01.npz")

X_train = data["X"]
y_train = data["y"]
epsilon = data["epsilon"]

mean = X_train.mean(axis=0)
std = X_train.std(axis=0)

X_train_standardized = (X_train - mean)/std

In [ ]:
X_train_standardized = torch.tensor(X_train_standardized, dtype=torch.float32).float()
y_train = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1).float()  # add dimension if needed

train_ds = TensorDataset(X_train_standardized, y_train)
train_dataloader = DataLoader(train_ds, batch_size=64)

In [ ]:
M = 12
seeds = [i for i in range(M)]

radii = np.linspace(0.2, 1.8, 40)
loss_fn = nn.MSELoss()

In [ ]:
def point_generator(radius, n=32):
    angles = np.linspace(0, 2*np.pi, n, endpoint=False)
    return np.stack([radius * np.cos(angles)], [radius * np.sin(angles)], axis=1)

In [ ]:
all_LEs = []
for seed in seeds:
    torch.manual_seed(seed)

    neural_ode = NeuralODEClassifier(input_dim=2, hidden_dim=3)
    neural_ode.apply(lambda m: init_weights(m, init_type="orthogonal", gain=0.9))

    train(neural_ode, train_dataloader, loss_fn, seed)

    run_LEs = []

    for r in radii:
        X = point_generator(r, n=32)
        X = torch.tensor(X, dtype=torch.float32)

        le_r = lyapunov_autograd(X[0, :], X[1, :], neural_ode, mean, std)

        run_LEs.append(le_r.mean())

    all_LEs.append(run_LEs)

all_LEs = np.array(all_LEs)

In [ ]:
mean_le = all_LEs.mean(axis=0)
std_le = all_LEs.std(axis=0)

plt.plot(radii, mean_le, label="Mean LE")
plt.fill_between(radii, mean_le-std_le, mean_le+std_le, alpha=0.3)
plt.axhline(0, linestyle="--", color="black")
plt.axvline(1.0, linestyle=":", color="red", label="Boundary")
plt.legend()
plt.xlabel("Radius")
plt.ylabel("Largest Lyapunov Exponent")